# 03 单序列消融实验

这本 Notebook 分别使用 T1-FLAIR、T2 和 DWI 训练三组单序列模型，并与 02 的三序列模型比较。三组实验使用相同的开发患者、折号和训练设置，用来观察不同影像序列提供的预测信息。

每组仍保留三类人工病灶掩模。因此结果反映的是在病灶信息相同的条件下，替换 MRI 序列带来的差异。运行前需要完成 00；比较三序列基线时，还需要 02 的开发预测结果。

下面先找到配置文件，读取这次实验的目录。


In [ ]:
from pathlib import Path

HERE = Path.cwd().resolve()
CANDIDATES = [HERE, HERE / "1117", HERE.parent / "1117"]
NB_ROOT = next((p for p in CANDIDATES if (p / "config.json").is_file()), None)
if NB_ROOT is None:
    raise FileNotFoundError("请从保存config.json的Notebook目录打开；不需要外层code目录。")


## 实验配置与文件记录

这一组函数负责管理实验文件。`load_config` 读取 `config.json`，`paths` 整理缓存、模型和报告的保存位置；后面的单元通过 `cfg` 读取参数，通过 `p` 查找文件。

`read_cohort` 读取患者清单，`development` 从中取出开发队列。保存结果时，`record` 会同时记录配置和代码版本，便于回查某次结果是怎样得到的。`fresh` 用来检查目标文件是否已存在，重新开展一组实验时可以换一个 `run_id`。


In [ ]:
"""Configuration, provenance and stage boundaries; standard library only."""
from __future__ import annotations

import hashlib
import json
import re
from pathlib import Path

SEED = 20260922
CLINICAL = ["age", "education", "nihss", "hcy", "ua"]
SEQUENCES = ["t1", "t2", "dwi"]
IMAGE_FILES = ["t1_flair.nii.gz", "t2.nii.gz", "dwi_b1000.nii.gz"]


def load_config(path: str | Path) -> dict:
    path = Path(path).resolve()
    cfg = json.loads(path.read_text(encoding="utf-8"))
    if not re.fullmatch(r"[A-Za-z0-9_-]+", cfg["run_id"]):
        raise ValueError("run_id 只使用字母、数字、下划线或短横线。")
    cfg["_root"] = path.parent
    for key in ("excel_path", "patients_root", "existing_cache", "outputs_root"):
        if cfg.get(key):
            p = Path(cfg[key]).expanduser()
            cfg[key] = p if p.is_absolute() else path.parent / p
    cfg["_run"] = cfg["outputs_root"] / cfg["run_id"]
    if cfg.get("data_access") == "classification_cache_only":
        verify_classification_bundle(cfg)
    return cfg


def paths(cfg: dict) -> dict[str, Path]:
    root = cfg["_run"]
    return {
        "run": root, "data": root / "data", "cache": Path(cfg.get("existing_cache") or root / "data" / "resnet50_manual_cache.npz"),
        "adapter": root / "data" / "clinical_adapter.xlsx", "cohort": root / "data" / "cohort.csv",
        "temporal": root / "temporal", "full_cv": root / "full_cv", "single": root / "single_sequence",
        "radiomics": root / "radiomics", "explain": root / "explain", "segmentation": root / "segmentation",
        "reports": root / "reports",
    }


def dump_json(path: Path, obj) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=str, allow_nan=False), encoding="utf-8")


def fresh(*targets: Path) -> None:
    exists = [str(p) for p in targets if p.exists()]
    if exists:
        raise FileExistsError("产物已存在，请加载已有结果，或更换 run_id：" + ", ".join(exists))


def digest(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def source_snapshot(cfg: dict) -> dict:
    """Hash the literal definition cells, independently of outputs and execution counts."""
    result = {}
    for path in sorted(cfg["_root"].glob("*.ipynb")):
        notebook = json.loads(path.read_text(encoding="utf-8"))
        definitions = ["".join(cell["source"]) for cell in notebook["cells"]
                       if cell["cell_type"] == "code" and cell.get("metadata", {}).get("embedded_definition")]
        result[path.name] = hashlib.sha256("\n\n".join(definitions).encode("utf-8")).hexdigest()
    return result


def record(cfg: dict, stage: str, extra: dict | None = None) -> None:
    obj = {"stage": stage, "config": {k: v for k, v in cfg.items() if not k.startswith("_")},
           "notebook_definitions_sha256": source_snapshot(cfg),
           "protocol": "original_resnet_integration_no_architecture_optimization", **(extra or {})}
    dump_json(paths(cfg)["run"] / "provenance" / f"{stage}.json", obj)


def read_cohort(cfg: dict):
    import pandas as pd
    p = paths(cfg)["cohort"]
    if not p.exists():
        raise FileNotFoundError("先运行00的数据准备与病例清单单元。")
    table = pd.read_csv(p, dtype={"mr": str})
    if not table["mr"].is_unique or not set(table["split"]) <= {"development", "validation"}:
        raise ValueError("病例清单编号重复或队列标记无效。")
    return table


def development(cfg: dict):
    return read_cohort(cfg).query("split == 'development'").reset_index(drop=True)


def require_input(cfg: dict, key: str) -> Path:
    value = cfg.get(key)
    if not value or not Path(value).exists():
        raise FileNotFoundError(f"请在 config.json 配置有效的 {key}。")
    return Path(value)


def verify_classification_bundle(cfg: dict) -> None:
    """Verify portable data without accessing the original image directories."""
    root = cfg["_root"].resolve()
    manifest_path = root / "classification_bundle.json"
    if not manifest_path.is_file():
        raise FileNotFoundError("分类数据包缺少 classification_bundle.json，请上传完整目录。")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("kind") != "classification_cache_only" or manifest["run_id"] != cfg["run_id"]:
        raise ValueError("分类数据包与当前 run_id 不一致。")
    p = paths(cfg)
    for key in ("cache", "cohort"):
        expected = (root / manifest[key]).resolve()
        expected.relative_to(root)
        if p[key].resolve() != expected or manifest[key] not in manifest["artifacts"]:
            raise ValueError(f"分类数据包的 {key} 路径或清单被改变。")
    for name, expected_hash in manifest["artifacts"].items():
        file = (root / name).resolve()
        file.relative_to(root)
        if not file.is_file() or digest(file) != expected_hash:
            raise ValueError(f"分类数据包文件缺失或校验失败：{name}")
    if manifest["notebook_definitions_sha256"] != source_snapshot(cfg):
        raise ValueError("Notebook 定义与数据包导出时不同，请使用对应版本或重新导出。")


def require_native_images(cfg: dict) -> None:
    if cfg.get("data_access") == "classification_cache_only":
        raise ValueError("本数据包只有分类缓存，没有原分辨率影像；影像组学和分割需另外准备原生数据。")


In [ ]:
cfg = load_config(NB_ROOT / "config.json")
p = paths(cfg)
print("实验：", cfg["run_id"])
print("结果目录：", p["run"])


## 表格与数组处理

NumPy 用于处理影像数组和预测概率，pandas 用于读取患者清单、合并临床信息及整理结果表。JSON 用于保存参数和指标；正则表达式用于检查编号等文本字段。

ROC、混淆矩阵和重采样工具用于整理各组指标，并比较同一批患者上的预测差异。


In [ ]:
from __future__ import annotations
import json
import re
import numpy as np
import pandas as pd


In [ ]:
from sklearn.metrics import brier_score_loss, confusion_matrix, roc_auc_score, roc_curve
from sklearn.utils import resample


## 单序列训练的工具与设置

这里加载 ResNet50、图像增强和交叉验证所需的工具，设置随机种子、每批患者数与计算设备。`SINGLE_SEQUENCES` 记录三个序列在缓存中的通道位置：T1-FLAIR 为 0，T2 为 1，DWI 为 2。

后面的实验循环通过这个映射，每次只取一个影像通道。


In [ ]:
from __future__ import annotations
import copy
import random
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.transforms.functional import affine
from torchvision.transforms import InterpolationMode

SEED = 20260922
BATCH_SIZE = 16
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SINGLE_SEQUENCES = {"t1": 0, "t2": 1, "dwi": 2}
CACHE = OUT_DIR = TEMPORAL_OUT = TEMPORAL_SPLIT = None
PREP_BASE = PREP_OUT = ADAPTER_XLSX = None


## 连接输入与输出目录

`configure_baseline` 把配置中确定的文件位置传给后续函数，包括影像缓存、临床表、时间分组表和模型输出目录。后面调用数据处理或模型函数时，就会使用同一组路径。


In [ ]:
from __future__ import annotations

def configure_baseline(cache, full_cv, temporal, adapter, patients_root=None):
    """Configure only input/output locations; do not alter the training recipe."""
    global CACHE, OUT_DIR, TEMPORAL_OUT, TEMPORAL_SPLIT
    global PREP_BASE, PREP_OUT, ADAPTER_XLSX
    CACHE, OUT_DIR, TEMPORAL_OUT = Path(cache), Path(full_cv), Path(temporal)
    TEMPORAL_SPLIT, ADAPTER_XLSX = TEMPORAL_OUT / "temporal_split.csv", Path(adapter)
    PREP_BASE, PREP_OUT = Path(patients_root) if patients_root else None, CACHE


## 病灶分支、数据增强与概率输出

`MaskEncoder` 将每层的三类病灶掩模编码为 256 维特征，三组单序列实验使用相同的病灶分支。

`augment_batch` 对同一患者的四层影像和掩模同步进行几何变换，再对影像加入亮度和噪声变化。`seed_everything` 设置随机数来源，`predict_proba` 则切换到评估状态，分批返回患者的阳性概率。


In [ ]:
from __future__ import annotations


# 直接复制：code/train_resnet50_manual_cv.py :: seed_everything
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# 直接复制：code/train_resnet50_manual_cv.py :: MaskEncoder
class MaskEncoder(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.features(x.float())


# 直接复制：code/train_resnet50_manual_cv.py :: augment_batch
def augment_batch(images: torch.Tensor, masks: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    out_images = images.clone()
    out_masks = masks.clone()
    B, S = images.shape[:2]
    for b in range(B):
        angle = random.uniform(-6.0, 6.0)
        translate = (random.uniform(-6.0, 6.0), random.uniform(-6.0, 6.0))
        scale = random.uniform(0.96, 1.04)
        for s in range(S):
            out_images[b, s] = affine(
                images[b, s],
                angle=angle,
                translate=translate,
                scale=scale,
                shear=0.0,
                interpolation=InterpolationMode.BILINEAR,
                fill=0.0,
            )
            out_masks[b, s] = affine(
                masks[b, s].float(),
                angle=angle,
                translate=translate,
                scale=scale,
                shear=0.0,
                interpolation=InterpolationMode.NEAREST,
                fill=0.0,
            ).byte()

    jitter = 0.9 + 0.2 * torch.rand(B, 1, 1, 1, 1, device=out_images.device)
    out_images = out_images * jitter
    if random.random() < 0.5:
        out_images = out_images + torch.randn_like(out_images) * 0.008
    out_images = out_images.clamp(0.0, 1.0)
    return out_images, out_masks


# 直接复制：code/train_resnet50_manual_cv.py :: predict_proba
@torch.no_grad()
def predict_proba(
    model: nn.Module,
    X: torch.Tensor,
    M: torch.Tensor,
    batch_size: int = BATCH_SIZE,
) -> np.ndarray:
    model.eval()
    outs = []
    for i in range(0, len(X), batch_size):
        xb = X[i : i + batch_size].to(DEVICE)
        mb = M[i : i + batch_size].to(DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
            outs.append(torch.sigmoid(model(xb, mb)).float().cpu().numpy())
    return np.concatenate(outs)


## 单通道 ResNet 与每折训练

`SingleSequenceDualBranchModel` 把 ResNet50 的第一层卷积改为单通道输入，初始化时对预训练卷积的三个输入通道权重取均值。其余部分继续按每层 2048 维影像特征、256 维病灶特征和四层注意力汇总的方式计算。

`train_single_sequence_fold` 完成一折训练，使用 AdamW、加权二元交叉熵和数据增强。最多训练 30 轮，连续 8 轮留出折 AUC 未改善时停止，返回该折 AUC 最佳时的模型。


In [ ]:
from __future__ import annotations


# 直接复制：code/train_resnet50_single_sequence_manual_cv.py :: SingleSequenceDualBranchModel
class SingleSequenceDualBranchModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        old_conv = backbone.conv1
        conv1 = nn.Conv2d(
            1,
            old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            bias=False,
        )
        with torch.no_grad():
            conv1.weight.copy_(old_conv.weight.mean(dim=1, keepdim=True))
        backbone.conv1 = conv1
        self.image_features = nn.Sequential(*list(backbone.children())[:-1])
        self.mask_features = MaskEncoder()
        self.attention = nn.Sequential(
            nn.Linear(2304, 256),
            nn.Tanh(),
            nn.Dropout(0.2),
            nn.Linear(256, 1),
        )
        self.head = nn.Sequential(
            nn.LayerNorm(2304),
            nn.Dropout(0.4),
            nn.Linear(2304, 1),
        )

    def forward(self, images: torch.Tensor, masks: torch.Tensor) -> torch.Tensor:
        B, S, C, H, W = images.shape
        im = images.view(B * S, C, H, W)
        image_feat = self.image_features(im)
        image_feat = torch.flatten(image_feat, 1)
        image_feat = image_feat.view(B, S, -1)

        mask = masks.view(B * S, 3, H, W)
        mask_feat = self.mask_features(mask)
        mask_feat = mask_feat.view(B, S, -1)

        feat = torch.cat([image_feat, mask_feat], dim=-1)
        attn = self.attention(feat).squeeze(-1)
        attn = torch.softmax(attn, dim=1)
        pooled = (feat * attn.unsqueeze(-1)).sum(dim=1)
        return self.head(pooled).squeeze(-1)


# 直接复制：code/train_resnet50_single_sequence_manual_cv.py :: train_fold
def train_single_sequence_fold(
    Xtr: torch.Tensor,
    Mtr: torch.Tensor,
    ytr: torch.Tensor,
    Xva: torch.Tensor,
    Mva: torch.Tensor,
    yva: torch.Tensor,
    fold: int,
) -> SingleSequenceDualBranchModel:
    model = SingleSequenceDualBranchModel().to(DEVICE)
    optimizer = torch.optim.AdamW(
        [
            {"params": model.image_features.parameters(), "lr": 1e-4},
            {
                "params": list(model.mask_features.parameters())
                + list(model.attention.parameters())
                + list(model.head.parameters()),
                "lr": 5e-4,
            },
        ],
        weight_decay=1e-4,
    )
    n_pos = int((ytr > 0.5).sum())
    n_neg = len(ytr) - n_pos
    pos_weight = torch.tensor([max(n_neg / max(n_pos, 1), 0.5)], device=DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())
    best_auc, best_state, patience = 0.0, None, 0

    for epoch in range(1, 31):
        t0 = time.time()
        model.train()
        perm = torch.randperm(len(Xtr))
        epoch_loss = 0.0
        for i in range(0, len(Xtr), BATCH_SIZE):
            bi = perm[i : i + BATCH_SIZE]
            xb, mb = Xtr[bi], Mtr[bi]
            xb, mb = augment_batch(xb, mb)
            xb = xb.to(DEVICE)
            mb = mb.to(DEVICE)
            yb = ytr[bi].to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
                loss = criterion(model(xb, mb), yb)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            epoch_loss += float(loss.detach()) * len(bi)
        epoch_loss /= len(Xtr)

        val_pred = predict_proba(model, Xva, Mva)
        val_auc = float(roc_auc_score(yva.numpy(), val_pred))
        if val_auc > best_auc:
            best_auc, best_state, patience = val_auc, copy.deepcopy(model.state_dict()), 0
        else:
            patience += 1
        if epoch % 3 == 0 or patience >= 8:
            print(
                f"fold {fold} epoch {epoch:02d} loss={epoch_loss:.4f} "
                f"val_auc={val_auc:.4f} best={best_auc:.4f} time={time.time()-t0:.1f}s",
                flush=True,
            )
        if patience >= 8:
            break

    model.load_state_dict(best_state)
    return model


## 指标计算与同一患者上的配对比较

前两个函数负责从开发预测选择分类阈值，并计算 AUC、准确率、敏感度和特异度。`paired_comparison` 先按 MR 编号对齐两个模型的患者和标签，再在每次抽样中同时取出两个模型对同一批患者的概率，计算 AUC 差值。

输出的差值按“模型 A 减模型 B”定义，同时给出重采样区间和经验 p 值。配对方式保留了两个模型在同一患者上预测的关联。


In [ ]:
from __future__ import annotations

def threshold_from_development(y, probability):
    fpr, tpr, thresholds = roc_curve(y, probability)
    return float(thresholds[np.argmax(tpr - fpr)])


def classification_metrics(y, probability, threshold, n_bootstrap=1000):
    y, probability = np.asarray(y, dtype=int), np.asarray(probability, dtype=float)
    if len(y) != len(probability) or set(y) != {0, 1} or not np.isfinite(probability).all():
        raise ValueError("标签需包含两类，概率必须有限且与患者一一对应。")
    if ((probability < 0) | (probability > 1)).any():
        raise ValueError("概率不在[0,1]。")
    tn, fp, fn, tp = confusion_matrix(y, probability >= threshold, labels=[0, 1]).ravel()
    result = {"n": len(y), "auc": float(roc_auc_score(y, probability)), "threshold": float(threshold),
              "accuracy": float((tp + tn) / len(y)), "sensitivity": float(tp / (tp + fn)),
              "specificity": float(tn / (tn + fp)), "brier": float(brier_score_loss(y, probability)),
              "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn)}
    if n_bootstrap:
        values = []
        for seed in range(n_bootstrap):
            idx = resample(np.arange(len(y)), replace=True, stratify=y, random_state=seed)
            values.append(roc_auc_score(y[idx], probability[idx]))
        result.update(auc_ci_low=float(np.percentile(values, 2.5)), auc_ci_high=float(np.percentile(values, 97.5)))
    return result


def paired_comparison(a: pd.DataFrame, b: pd.DataFrame, n_bootstrap=1000):
    for frame in (a, b):
        if not frame["mr"].is_unique or set(frame["label"]) != {0, 1}:
            raise ValueError("比较需要唯一患者编号与两类结局。")
    if set(a["mr"]) != set(b["mr"]):
        raise ValueError("两组患者不完全一致；不静默取交集。")
    merged = a.merge(b, on="mr", suffixes=("_a", "_b"), validate="one_to_one")
    if not (merged.label_a == merged.label_b).all():
        raise ValueError("同一患者的标签不一致。")
    y, pa, pb = merged.label_a.to_numpy(), merged.probability_a.to_numpy(), merged.probability_b.to_numpy()
    if not np.isfinite(pa).all() or not np.isfinite(pb).all():
        raise ValueError("预测包含非有限值。")
    differences = []
    for seed in range(n_bootstrap):
        idx = resample(np.arange(len(y)), replace=True, stratify=y, random_state=seed)
        differences.append(roc_auc_score(y[idx], pa[idx]) - roc_auc_score(y[idx], pb[idx]))
    d = np.asarray(differences)
    return {"auc_difference": float(roc_auc_score(y, pa) - roc_auc_score(y, pb)),
            "ci_low": float(np.percentile(d, 2.5)), "ci_high": float(np.percentile(d, 97.5)),
            "empirical_p": min(1.0, 2 * min(float((d <= 0).mean()), float((d >= 0).mean())))}


## 依次完成三组单序列实验

`run_single_sequence_ablation` 从缓存中取出开发患者，按 T1-FLAIR、T2、DWI 依次训练。每组截取一个影像通道，病灶掩模保持三个通道，并按已保存的五折划分调用训练函数。

每个序列目录下保存五个折模型和开发折外概率，汇总表保存各折平均 AUC、合并 AUC 及阈值相关指标，供后面与三序列模型比较。


In [ ]:
from __future__ import annotations

def configure(cfg):
    p = paths(cfg)
    configure_baseline(p["cache"], p["full_cv"], p["temporal"], p["adapter"], cfg.get("patients_root"))
    return p


def run_single_sequence_ablation(cfg):
    """Same original single-sequence model and train_fold, restricted to development."""
    import torch
    from sklearn.metrics import roc_auc_score
    p = configure(cfg)
    fresh(p["single"])
    dev = development(cfg)
    indices = dev.cache_index.to_numpy(dtype=int)
    with np.load(p["cache"], allow_pickle=False) as cache:
        X = torch.from_numpy(cache["images"][indices].astype(np.float32))
        M = torch.from_numpy(cache["masks"][indices].astype(np.uint8))
    y = dev.label.to_numpy(dtype=int)
    summaries = []
    for sequence, channel in SINGLE_SEQUENCES.items():
        seed_everything(SEED)
        output = p["single"] / sequence
        output.mkdir(parents=True)
        x = X[:, :, channel:channel + 1].contiguous()
        oof, aucs = np.zeros(len(y), dtype=np.float32), []
        for fold in range(1, 6):
            va = np.flatnonzero(dev.development_fold.to_numpy() == fold)
            tr = np.flatnonzero(dev.development_fold.to_numpy() != fold)
            model = train_single_sequence_fold(x[tr], M[tr], torch.tensor(y[tr], dtype=torch.float32),
                                      x[va], M[va], torch.tensor(y[va], dtype=torch.float32), fold)
            oof[va] = predict_proba(model, x[va], M[va])
            aucs.append(float(roc_auc_score(y[va], oof[va])))
            torch.save(model.state_dict(), output / f"fold_{fold}.pt")
        threshold = threshold_from_development(y, oof)
        table = dev[["mr", "label", "development_fold"]].copy()
        table["probability"] = oof
        table.to_csv(output / "development_oof.csv", index=False)
        summaries.append({"sequence": sequence, "mean_fold_auc": float(np.mean(aucs)),
                          **classification_metrics(y, oof, threshold, n_bootstrap=0)})
    summary = pd.DataFrame(summaries)
    summary.to_csv(p["single"] / "summary.csv", index=False)
    record(cfg, "single_sequence", {"scope": "development", "manual_mask_channels": 3})
    return summary


## 1. 运行三组单序列训练

启用后会依次完成三个序列、每个序列五折的训练。结果写入 `single_sequence/t1/`、`t2/` 和 `dwi/`，训练结束后返回三组汇总指标。


In [ ]:
RUN_SINGLE_SEQUENCE = False

if RUN_SINGLE_SEQUENCE:
    summary = run_single_sequence_ablation(cfg)
    display(summary)


## 2. 与三序列模型放在同一张表中比较

读取 02 的三序列开发结果，再加入三组单序列结果。重点比较相同口径的 AUC，同时查看折间波动，判断多序列联合是否带来稳定收益。


In [ ]:
import json
import pandas as pd
rows = []
if (p["temporal"] / "development_cv_metrics.json").exists():
    baseline = json.loads((p["temporal"] / "development_cv_metrics.json").read_text(encoding="utf-8"))
    rows.append({"sequence": "t1+t2+dwi", "mean_fold_auc": baseline["mean_auc"], "auc": baseline["pooled_auc"]})
if (p["single"] / "summary.csv").exists():
    rows.extend(pd.read_csv(p["single"] / "summary.csv").to_dict("records"))
if rows:
    display(pd.DataFrame(rows))


## 3. 比较三序列与单序列的 AUC 差值

下面将三序列模型设为 A，分别将每个单序列模型设为 B，比较同一批开发患者的折外概率。差值为正表示三序列 AUC 更高；区间较宽则提示当前样本对差异的估计不够精确。

这些预测来自开发阶段，且各折曾据留出折 AUC 选择训练轮数，因此这里主要用于分析序列贡献，泛化能力仍需另做独立验证。


In [ ]:
COMPARE_DEVELOPMENT_OOF = False
if COMPARE_DEVELOPMENT_OOF:

    baseline = pd.read_csv(p["temporal"] / "development_oof.csv").rename(columns={"dev_oof_prob": "probability"})
    differences = []
    for sequence in ["t1", "t2", "dwi"]:
        single = pd.read_csv(p["single"] / sequence / "development_oof.csv")
        differences.append({"a": "three_sequence", "b": sequence, **paired_comparison(baseline, single)})
    display(pd.DataFrame(differences))


## 后续结构消融可以回答什么

当前三组实验比较的是影像序列。若要分析病灶分支和注意力模块的作用，还需要分别训练去掉病灶分支、或用平均汇总替换注意力的模型，并保持其他条件一致。

直接把已训练模型的掩模输入置零，可以观察它对输入变化的敏感程度；删除病灶分支并重新训练，才能评价该结构在学习过程中的作用。
